In [ ]:
from mendeleev.fetch import fetch_table
import psi4
from mendeleev import element

In [ ]:
ptable = fetch_table("elements")

In [ ]:
with open('diatomics.txt','r') as f:
    diatomics = [i.strip().split(',') for i in f.readlines()]

In [ ]:
# Process and correct multiplicities
for A, B in diatomics:
    covrad = (element(A).covalent_radius_cordero + element(B).covalent_radius_cordero)/100
    xyz=f"""{A} 0 0 0
{B} 0 0 {covrad}
"""
    psi4.core.set_num_threads(12)   
    psi4.set_memory("10GB")
    psi4.core.be_quiet()
    geom = psi4.geometry(xyz)
    psi4.set_options({'reference': 'UKS',
                      'basis':'6-31G*',
                      'd_convergence':   1e-10,
                      'GEOM_MAXITER':    500,
                      'MAXITER':         500,
                      'r_convergence':   1e-5,
                      'e_convergence':   1e-4,
                     })
    DFT_energy, DFT_wfn = psi4.optimize('b3lyp',return_wfn=True)
    print(DFT_energy)
    # DFT_freq = psi4.frequency('oppbe',ref_wfn=DFT_wfn)
    
    
    xyz_str = "2\n"+DFT_wfn.molecule().save_string_xyz()
    print(xyz_str)
    if len(A)==2:
        xyz_str = xyz_str.replace(A.upper(),A)
    if len(B)==2:
        xyz_str = xyz_str.replace(B.upper(),B)        
    print(xyz_str)
    with open(f'{A}{B}.xyz', 'w') as f:
        f.write(xyz_str)